# 🧠 Checkpoint 9 — Un Réseau de Neurones Simple, Expliqué

## Module Data Science — Intelligence Artificielle

> **Prérequis :** cours *Fondamentaux IA/ML/DL* (`07-...md`).

Notebook **corrigé** du checkpoint 9. Objectif : **démystifier** le réseau de neurones —
montrer qu'il ne s'agit que de « somme pondérée + activation », ajusté par descente de gradient.

| Partie | Contenu |
|--------|---------|
| **A** | Un neurone en NumPy (de zéro) |
| **B** | Un réseau 3→4→1 (propagation avant) |
| **C** | La descente de gradient, à la main |
| **D** | Un vrai réseau avec `MLPClassifier` |

> ⚠️ Uniquement **NumPy + scikit-learn** (pas de TensorFlow/PyTorch) : on vise la compréhension des mécanismes.

In [ ]:
!pip install scikit-learn -q

import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-z))   # sortie dans [0, 1]
def relu(z):    return np.maximum(0, z)        # sortie dans [0, +inf[

print('Environnement prêt ✅')

---
# PARTIE A — Un neurone en NumPy (de zéro)

Un neurone fait **deux opérations** : une somme pondérée des entrées + biais (`z`), puis une activation (`a = f(z)`).

In [ ]:
x = np.array([0.5, 0.3, 0.2])     # entrées (features)
w = np.array([0.4, 0.7, 0.1])     # poids : importance de chaque entrée (APPRIS)
b = 0.1                            # biais : décalage / seuil ajustable

z = np.dot(w, x) + b               # 1) somme pondérée + biais
a = sigmoid(z)                     # 2) activation
print(f'z = {z:.3f}   ->   a = sigmoid(z) = {a:.3f}')

In [ ]:
# Effet d'un POIDS plus fort sur la 2e entrée
w_fort = np.array([0.4, 2.0, 0.1])
print('Sortie si w₂ passe de 0.7 à 2.0 :', round(sigmoid(np.dot(w_fort, x) + b), 3))

# Effet du BIAIS
print('Sortie si b = -1 (neurone moins facile à activer) :', round(sigmoid(np.dot(w, x) - 1), 3))
print('Sortie si b = +1 (neurone plus facile à activer)  :', round(sigmoid(np.dot(w, x) + 1), 3))

> 🔑 **Explication** : le **poids** dit combien chaque entrée compte (c'est ce que le réseau apprend) ;
> le **biais** rend le neurone plus ou moins facile à activer ; l'**activation** transforme `z` en un signal borné (ici une « probabilité »).

---
# PARTIE B — Un réseau à une couche cachée (forward)

Réseau **3 entrées → 4 neurones cachés (ReLU) → 1 sortie (sigmoïde)**.
Chaque couche applique : `a = f(W·x + b)`.

In [ ]:
# Poids/biais initialisés ALÉATOIREMENT -> réseau NON entraîné
W1 = np.random.randn(4, 3) * 0.5   # couche cachée : 4 neurones, 3 entrées
b1 = np.zeros(4)
W2 = np.random.randn(1, 4) * 0.5   # sortie : 1 neurone, 4 entrées
b2 = np.zeros(1)

def forward(x):
    a1 = relu(W1 @ x + b1)         # couche cachée (ReLU)
    y  = sigmoid(W2 @ a1 + b2)     # sortie (sigmoïde -> proba)
    return y, a1

x = np.array([0.5, 0.3, 0.2])
y, a1 = forward(x)
print('Activations de la couche cachée :', a1.round(3))
print('Prédiction (probabilité)        :', y.round(3))

> 🔑 **Explication** : la sortie n'a **aucun sens** ici car les poids sont **aléatoires**.
> Il faut **entraîner** le réseau (ajuster W1, W2, b1, b2) pour qu'elle devienne pertinente — c'est l'objet de la Partie C.

---
# PARTIE C — Comprendre l'apprentissage : la descente de gradient

Sur un neurone à **une entrée** (`y = sigmoid(w·x + b)`), on ajuste `w` et `b` pour
atteindre une cible. On calcule le **gradient** de la loss et on descend.

Dérivée en chaîne pour `L = (a - y)²`, `a = σ(z)`, `z = wx + b` :
`dL/dw = 2(a-y)·a(1-a)·x` et `dL/db = 2(a-y)·a(1-a)`.

In [ ]:
x, y_vrai = 1.5, 1.0      # on veut que le neurone sorte ~1 pour x=1.5
w, b = 0.0, 0.0           # poids initiaux
eta = 0.5                 # learning rate (taille du pas)

pertes = []
for i in range(30):
    z = w * x + b
    a = sigmoid(z)                    # prédiction
    loss = (a - y_vrai) ** 2          # erreur quadratique
    pertes.append(loss)

    d = 2 * (a - y_vrai) * a * (1 - a)   # facteur commun (dérivée en chaîne)
    dw, db = d * x, d
    w -= eta * dw                     # descente de gradient
    b -= eta * db

print(f'Poids appris     : w = {w:.3f}, b = {b:.3f}')
print(f'Prédiction finale: {sigmoid(w*x + b):.3f}   (cible = {y_vrai})')
print(f'Loss : {pertes[0]:.3f} (départ)  ->  {pertes[-1]:.4f} (fin)')

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(pertes, marker='o')
plt.xlabel('Itération'); plt.ylabel('Loss')
plt.title('La loss diminue → le neurone apprend'); plt.grid(alpha=.3)
plt.show()

> 🔑 **Explication** : à chaque pas, on corrige `w` et `b` dans le sens qui **réduit l'erreur**.
> La courbe descend : c'est exactement ce que fait un réseau — mais sur des millions de poids à la fois, via la **backpropagation**.

---
# PARTIE D — Un vrai réseau avec `MLPClassifier`

On entraîne un réseau opérationnel sur un dataset réel (**Breast Cancer**) et on l'évalue.
⚠️ La **standardisation** est indispensable pour un réseau (descente de gradient sensible à l'échelle).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# 1. Données + split stratifié
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Standardisation (fit sur le TRAIN uniquement -> anti-fuite)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# 3. Réseau : 2 couches cachées (16 puis 8 neurones), activation ReLU
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu',
                    max_iter=500, random_state=42)
mlp.fit(X_tr_s, y_tr)
print('Réseau entraîné ✅')

In [ ]:
# 4. Évaluation
y_pred = mlp.predict(X_te_s)
print(f'Accuracy : {accuracy_score(y_te, y_pred):.3f}')
print(f'F1       : {f1_score(y_te, y_pred):.3f}')
print(f'Couches (entrée+cachées+sortie) : {mlp.n_layers_}')
print(f'Neurones cachés : {mlp.hidden_layer_sizes}')

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['malin', 'bénin']).plot(cmap='Blues')
plt.title('Matrice de confusion'); plt.show()

In [ ]:
# 5. Courbe de loss : la descente de gradient en action
plt.figure(figsize=(6, 4))
plt.plot(mlp.loss_curve_)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Descente de la loss pendant l\'entraînement'); plt.grid(alpha=.3)
plt.show()
print(f'Loss finale : {mlp.loss_curve_[-1]:.4f}')

> 🔑 **Explication** : le réseau a **2 couches cachées** (16 puis 8 neurones) + entrée + sortie = 4 couches.
> La courbe de loss **descend puis se stabilise** = convergence réussie. Sans l'étape 2 (standardisation),
> l'entraînement serait beaucoup plus lent et instable.

---
# ✅ Synthèse du checkpoint

| Partie | Ce que vous avez démontré |
|--------|---------------------------|
| **A** | Un neurone = somme pondérée + activation ; rôle des poids/biais |
| **B** | Un réseau enchaîne des couches (forward) ; sans entraînement, la sortie est aléatoire |
| **C** | L'apprentissage = descente de gradient qui réduit la loss |
| **D** | Un réseau réel s'entraîne, se standardise, s'évalue, et sa loss converge |

> 🔑 **En une phrase** : un réseau de neurones n'est qu'un empilement de « somme pondérée + activation »
> dont on ajuste les poids par descente de gradient pour minimiser une erreur — répété à grande échelle.

### 🚀 Pour aller plus loin
- Changez `hidden_layer_sizes` (ex. `(64, 32)`) et observez l'effet sur la loss et l'accuracy.
- Comparez `activation='relu'` vs `'tanh'`.
- Remplacez le neurone de la Partie C par 2 entrées et généralisez la descente de gradient.

---
*📘 Module Data Science — Checkpoint 9 : Réseau de Neurones Simple | Bootcamp Data Science*